In [2]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
import yaml

# Project paths (the notebook lives in notebooks/)
ROOT = Path("..")
RAW_OSM = ROOT / "data" / "raw" / "osm"
INTERIM = ROOT / "data" / "interim"
RAW_OSM.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)

with open(ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)

COUNTRIES = config["scope"]["countries"]
PAUSE_S = config["osm"]["request_pause_s"]
print(len(COUNTRIES), "countries | pause:", PAUSE_S, "s")

30 countries | pause: 5 s


In [3]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def build_query(country_code: str) -> str:
    """Overpass QL query for data centres in one country (existing, under construction, planned)."""
    return f"""
    [out:json][timeout:180];
    area["ISO3166-1"="{country_code}"][admin_level=2]->.country;
    (
      nwr["telecom"="data_center"](area.country);
      nwr["building"="data_center"](area.country);
      nwr["industrial"="data_centre"](area.country);
      nwr["construction:telecom"="data_center"](area.country);
      nwr["construction"="data_center"](area.country);
      nwr["proposed:telecom"="data_center"](area.country);
    );
    out geom;
    """

print(build_query("PT"))


    [out:json][timeout:180];
    area["ISO3166-1"="PT"][admin_level=2]->.country;
    (
      nwr["telecom"="data_center"](area.country);
      nwr["building"="data_center"](area.country);
      nwr["industrial"="data_centre"](area.country);
      nwr["construction:telecom"="data_center"](area.country);
      nwr["construction"="data_center"](area.country);
      nwr["proposed:telecom"="data_center"](area.country);
    );
    out geom;
    


In [4]:
HEADERS = {"User-Agent": "ia-agua-datacenters (portfolio project; github.com/Carol-Sust/Sustainability)"}

def fetch_country(country_code: str, force: bool = False, max_retries: int = 4) -> dict:
    """Download OSM data centres for one country, caching the raw JSON in data/raw/osm/."""
    cache_file = RAW_OSM / f"{country_code}.json"
    if cache_file.exists() and not force:
        return json.loads(cache_file.read_text())

    query = build_query(country_code)
    for attempt in range(1, max_retries + 1):
        response = requests.post(OVERPASS_URL, data={"data": query}, headers=HEADERS, timeout=300)
        if response.status_code == 200:
            data = response.json()
            cache_file.write_text(json.dumps(data))
            return data
        # 429 = too many requests, 504 = server busy -> wait and retry
        wait = 30 * attempt
        print(f"{country_code}: HTTP {response.status_code}, retry {attempt}/{max_retries} in {wait}s")
        time.sleep(wait)

    raise RuntimeError(f"{country_code}: failed after {max_retries} attempts")

In [5]:
pt = fetch_country("PT")
elements = pt["elements"]
print("Elements found:", len(elements))

# How many points (node), polygons (way) and relations?
print(pd.Series([e["type"] for e in elements]).value_counts())

# Tags of each element as a table
tags = pd.json_normalize([e.get("tags", {}) for e in elements])
cols = [c for c in ["name", "operator", "telecom", "building", "construction", "proposed:telecom"] if c in tags.columns]
tags[cols].head(20)

Elements found: 19
way     15
node     4
Name: count, dtype: int64


,name,operator,telecom,building
0,NaN,MEO,data_center,NaN
1,Centro de Computação Avançada do Minho,Fundação para a Ciência e Tecnologia,data_center,NaN
2,NaN,NaN,data_center,NaN
3,NaN,NaN,data_center,NaN
4,Equinix LS1,Equinix,data_center,data_center
5,NOS Imopolis II,NOS,data_center,data_center
6,NOS Imopolis I,NOS,data_center,data_center
7,Seixal Colocation Site,TATA Communications,data_center,yes
8,Data Center Portugal Telecom da Covilhã,Portugal Telecom,data_center,yes
9,Datacenter de Riba d'Ave,REN;NOS,data_center,data_center


In [6]:
from tqdm import tqdm

counts = {}
failed = []

for cc in tqdm(COUNTRIES):
    already_cached = (RAW_OSM / f"{cc}.json").exists()
    try:
        data = fetch_country(cc)
        counts[cc] = len(data["elements"])
    except RuntimeError as error:
        print(error)
        failed.append(cc)
    # Only pause after a real request, not when reading from cache
    if not already_cached:
        time.sleep(PAUSE_S)

print("Failed:", failed if failed else "none")

100%|██████████████████████████████████████████| 30/30 [00:52<00:00,  1.75s/it]

Failed: none


In [7]:
!git add -A
!git commit -m "Step 1: Overpass download for all countries"
!git push

[main 1cb359a] Step 1: Overpass download for all countries
 2 files changed, 60 insertions(+), 445 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 8 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 4.50 KiB | 4.50 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Carol-Sust/Sustainability.git
   4cbe18b..1cb359a  main -> main


In [8]:
saved = sorted(p.stem for p in RAW_OSM.glob("*.json"))
missing = sorted(set(COUNTRIES) - set(saved))
print(len(saved), "files | missing:", missing if missing else "none")

31 files | missing: none


In [9]:
counts = {cc: len(json.loads((RAW_OSM / f"{cc}.json").read_text())["elements"]) for cc in saved}
summary = pd.Series(counts, name="elements").sort_values(ascending=False)
print("Total elements:", summary.sum())
summary

Total elements: 1809


FR       390
DE       320
GB       310
NL       170
IE       113
IT        75
ES        52
BE        41
FI        38
CH        35
SE        35
PL        34
DK        26
CZ        24
HU        23
NO        21
PT        19
AT        17
HR        13
LU        10
GR        10
LV         6
RO         6
BG         6
LT         5
SK         5
SI         3
EE         2
MT         0
False      0
CY         0
Name: elements, dtype: int64

In [15]:
saved = sorted(p.stem for p in RAW_OSM.glob("*.json"))
counts = {cc: len(json.loads((RAW_OSM / f"{cc}.json").read_text())["elements"]) for cc in saved}
summary = pd.Series(counts, name="elements").sort_values(ascending=False)
print(len(saved), "countries | total elements:", summary.sum())
summary

30 countries | total elements: 1809


FR    390
DE    320
GB    310
NL    170
IE    113
IT     75
ES     52
BE     41
FI     38
SE     35
CH     35
PL     34
DK     26
CZ     24
HU     23
NO     21
PT     19
AT     17
HR     13
LU     10
GR     10
LV      6
RO      6
BG      6
LT      5
SK      5
SI      3
EE      2
MT      0
CY      0
Name: elements, dtype: int64

In [10]:
import geopandas as gpd
from shapely.geometry import Point, Polygon, LineString
from shapely.ops import unary_union

CRS_GEO = config["crs"]["geographic"]      # EPSG:4326
CRS_AREA = config["crs"]["equal_area"]     # EPSG:3035

In [11]:
def way_to_geometry(coords):
    """A list of (lon, lat) points -> Polygon if closed, otherwise LineString."""
    if len(coords) >= 4 and coords[0] == coords[-1]:
        return Polygon(coords)
    if len(coords) >= 2:
        return LineString(coords)
    return Point(coords[0])

def element_geometry(element):
    """Build a shapely geometry from an Overpass element (out geom)."""
    if element["type"] == "node":
        return Point(element["lon"], element["lat"])

    if element["type"] == "way":
        coords = [(p["lon"], p["lat"]) for p in element.get("geometry", [])]
        return way_to_geometry(coords) if coords else None

    if element["type"] == "relation":
        # Combine the outer members of the relation (e.g. a campus with several buildings)
        parts = []
        for member in element.get("members", []):
            if member.get("role") in ("outer", "") and "geometry" in member:
                coords = [(p["lon"], p["lat"]) for p in member["geometry"]]
                if coords:
                    parts.append(way_to_geometry(coords))
        return unary_union(parts) if parts else None

    return None

In [20]:
def element_status(tags):
    """existing / under_construction / planned, based on OSM lifecycle tags."""
    if tags.get("proposed:telecom") == "data_center" or tags.get("landuse") == "proposed":
        return "planned"
    if (
        tags.get("construction:telecom") == "data_center"
        or tags.get("construction") == "data_center"
        or tags.get("landuse") == "construction"
        or tags.get("building") == "construction"
    ):
        return "under_construction"
    return "existing"

In [21]:
rows = []
for cc in COUNTRIES:
    data = json.loads((RAW_OSM / f"{cc}.json").read_text())
    for element in data["elements"]:
        tags = element.get("tags", {})
        rows.append({
            "osm_type": element["type"],
            "osm_id": element["id"],
            "country": cc,
            "name": tags.get("name"),
            "operator": tags.get("operator"),
            "status": element_status(tags),
            "geometry": element_geometry(element),
        })

gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS_GEO)

# Remove elements without geometry and any duplicates
gdf = gdf.dropna(subset=["geometry"]).drop_duplicates(subset=["osm_type", "osm_id"])

# Area in m² (only for polygons; points and lines get NaN)
is_polygon = gdf.geom_type.isin(["Polygon", "MultiPolygon"])
gdf["area_m2"] = gdf.to_crs(CRS_AREA).area.where(is_polygon)

# Centroid as a point, computed in metres for accuracy and converted back to lat/lon
centroids = gdf.to_crs(CRS_AREA).centroid.to_crs(CRS_GEO)
gdf["lon"] = centroids.x
gdf["lat"] = centroids.y

print(len(gdf), "elements")
gdf.head()

1808 elements


,osm_type,osm_id,country,name,operator,status,geometry,area_m2,lon,lat
0,node,6578896177,AT,earthDATAsafe,Kapsch BusinessCom AG,existing,POINT (15.33817 47.44914),NaN,15.338173,47.449138
1,node,12596690617,AT,IT- und Data Center Linz,LINZ NETZ GmbH,existing,POINT (14.30832 48.28215),NaN,14.308318,48.282151
2,node,13092390748,AT,IT- und Data Center Linz,NaN,existing,POINT (14.3 48.3242),NaN,14.299996,48.324198
3,way,31990236,AT,Digital Realty VIE1 & VIE2,Digital Realty,existing,"POLYGON ((16.40878 48.26918, 16.40867 48.26874...",27721.168009,16.410523,48.268928
4,way,74662577,AT,Telekom-Tower,A1 / Telekom Austria,existing,"POLYGON ((15.43 47.06632, 15.42999 47.06619, 1...",870.136764,15.430370,47.066245


In [22]:
print(gdf["status"].value_counts(), "\n")
print(gdf.geom_type.value_counts(), "\n")
print(gdf["area_m2"].describe().round(0))

status
existing              1759
under_construction      38
planned                 11
Name: count, dtype: int64 

Polygon               1421
Point                  372
MultiLineString          9
MultiPolygon             5
GeometryCollection       1
Name: count, dtype: int64 

count      1426.0
mean      12128.0
std       47334.0
min           3.0
25%        1300.0
50%        3557.0
75%        8227.0
max      875069.0
Name: area_m2, dtype: float64


In [23]:
from shapely.ops import polygonize

def lines_to_polygon(geom):
    """Close line segments into polygons (for relations whose outline came in pieces)."""
    if geom.geom_type in ("Polygon", "MultiPolygon", "Point"):
        return geom
    parts = list(getattr(geom, "geoms", [geom]))
    lines = [p for p in parts if p.geom_type in ("LineString", "LinearRing", "MultiLineString")]
    polygons = [p for p in parts if p.geom_type in ("Polygon", "MultiPolygon")]
    polygons += list(polygonize(lines))
    return unary_union(polygons) if polygons else geom.centroid

gdf["geometry"] = gdf.geometry.apply(lines_to_polygon)

# Recompute area and centroid with the fixed geometries
is_polygon = gdf.geom_type.isin(["Polygon", "MultiPolygon"])
gdf["area_m2"] = gdf.to_crs(CRS_AREA).area.where(is_polygon)
centroids = gdf.to_crs(CRS_AREA).centroid.to_crs(CRS_GEO)
gdf["lon"], gdf["lat"] = centroids.x, centroids.y

print(gdf.geom_type.value_counts())

Polygon         1430
Point            372
MultiPolygon       6
Name: count, dtype: int64


In [24]:
cols = ["country", "name", "operator", "status", "osm_type", "area_m2"]
print("Largest:")
display(gdf.nlargest(10, "area_m2")[cols])
print("Smallest:")
display(gdf.nsmallest(10, "area_m2")[cols])

Largest:


,country,name,operator,status,osm_type,area_m2
166,FI,Alussuo,Tuike Finland Oy,under_construction,relation,3.312547e+06
1021,IE,Meta Clonee Data Center,NaN,existing,relation,1.368472e+06
142,FI,Google Haminan palvelinkeskus,Google,existing,way,8.750690e+05
32,BE,Google Data Center,Google,existing,way,8.467917e+05
1282,NL,Microsoft Datacenter,Microsoft,existing,way,4.996921e+05
1022,IE,Microsoft Grange Castle,Microsoft,existing,relation,4.976187e+05
1442,SE,EcoDataCenter,NaN,under_construction,relation,4.878489e+05
111,DK,Google Data Center Fredericia,Google,existing,way,4.788082e+05
1253,NL,Google Datacenter Eemshaven,Google,existing,way,4.341921e+05
1245,NL,Microsoft Datacenter West Europe,Microsoft,existing,way,3.530679e+05


Smallest:


,country,name,operator,status,osm_type,area_m2
1257,NL,NaN,NaN,existing,way,2.597401
1767,NO,NaN,NaN,existing,way,4.479545
1758,NO,Signal,NaN,existing,way,6.654209
1768,NO,NaN,NaN,existing,way,7.086897
1385,ES,NaN,NaN,existing,way,10.184668
1384,ES,centro de datos trasbase Negratín,NaN,existing,way,10.998098
1337,PT,NaN,NaN,existing,way,11.496762
660,DE,Dernbach 1,GasLINE,existing,way,12.238588
1354,SI,NaN,NaN,existing,way,12.676986
1070,IT,NaN,NaN,existing,way,12.925483


In [25]:
for threshold in [25, 50, 100, 200, 500]:
    n = (gdf["area_m2"] < threshold).sum()
    print(f"< {threshold:>3} m²: {n:>4} elements ({n / gdf['area_m2'].notna().sum():.1%} of polygons)")

<  25 m²:   21 elements (1.5% of polygons)
<  50 m²:   28 elements (1.9% of polygons)
< 100 m²:   52 elements (3.6% of polygons)
< 200 m²:   65 elements (4.5% of polygons)
< 500 m²:  139 elements (9.7% of polygons)


In [ ]:
cols = ["country", "name", "operator", "status", "area_m2"]
gdf[(gdf["area_m2"] >= 50) & (gdf["area_m2"] < 100)][cols].sort_values("area_m2")